# 🚀 Operational Flow:

**Executes the entire flow with a notebook interface:**
- Initiate DB
- Load CSV to DB
- Load (and validate) TAK repository
- Apply TAKs on DB
- View output

In [ ]:
# !pip install -r requirements-py313.txt  # use this on Python >= 3.8 (e.g. 3.13); requirements-py37.txt is pinned to old pandas/numpy with no py3.13 wheels
# !pip install -e .

In [ ]:
from pathlib import Path
from backend.dataaccess import DataAccess
from core.mediator import Mediator, setup_logging
import logging
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

In [ ]:
# Paths
KB_PATH = Path("core/knowledge-base")
DB_PATH = Path("backend/data/mediator.db")
# CSV_PATH = Path("backend/data/input_data.csv") # <- random generated data for testing
CSV_PATH = Path("backend/data/mimic-iv-input-data.csv") # <- real MIMIC-IV data (after preprocessing)

In [ ]:
# # 1. Connect to existing DB
# da = DataAccess(db_path=str(DB_PATH))

# 2. Or auto-create (and optionally drop existing)
da = DataAccess(db_path=str(DB_PATH), auto_create=True)

# Check stats
stats = da.get_table_stats()
for table, info in stats.items():
    print(f"{table}: {info['rows']} rows, {info['n_patients']} patients")

In [ ]:
# Load CSV into InputPatientData
total_rows = da.load_csv_to_input(
    csv_path=str(CSV_PATH),
    if_exists='append',           # 'append' or 'replace'
    clear_output_and_qa=False,    # Set True to clear outputs
    yes=True,                     # Auto-confirm
    column_mapping = {
            'PatientId': ['VisitId', 'PatientId'],
            'ConceptName': ['ConceptName'],
            'StartDateTime': ['StartDateTime', 'StartTime'],
            'EndDateTime': ['EndDateTime', 'EndTime'],
            'Value': ['Value']
        }                         # Match columns in CSV (values) to required expected names (keys)
)
print(f"Loaded {total_rows} rows")

In [ ]:
# Setup logging ONCE (before creating Mediator)
setup_logging(
    log_file=Path("logs/mediator.log"),
    console_level=logging.WARNING  # Only warnings/errors in console
)

# Initialize mediator
mediator = Mediator(knowledge_base_path=KB_PATH, data_access=da)

# Build TAK repository
repo = mediator.build_repository()

# List all TAK names
print("TAK Names:")
for tak_name in sorted(repo.taks.keys()):
    print(f"  - {tak_name}")

# Save repository to file (uncompressed)
repo.save("backend/data/tak-repo.pkl", compress=False)

# Save repository to exportable file (uncompressed)
repo.export_portable("backend/data/tak-repo-portable.json")

# # Export TAK metadata to Excel/CSV for documentation
# df_tak_metadata = repo.export_metadata()

# print(f"Exported metadata for {len(df_tak_metadata)} TAKs")

# # Save to Excel (preserves formatting better for JSON columns)
# excel_path = "backend/data/tak_metadata.xlsx"
# df_tak_metadata.to_excel(excel_path, index=False, engine='openpyxl')
# print(f"\n✅ Saved to: {excel_path}")

In [ ]:
from core.tak.raw_concept import RawConcept
from core.tak.pattern import Pattern
import sqlite3

# ▶ Knob 1: Patient subset  (None = all patients)
# patient_ids = [1000]
patient_ids = None

# ▶ Knob 2: TAK subset  (None = full run; list TAKs you changed for a partial re-run)
# tak_ids = ["INSULIN_ON_HYPERGLYCEMIA_PATTERN", "ROUTINE_GLUCOSE_MEASURE_ON_STEROIDS_PATTERN"]
tak_ids = None

# ============================================================

if tak_ids is not None:
    # --- Partial re-run ---

    # Walk transitive dependency closure using the pre-built repo graph
    def _get_transitive_deps(repo, root_names):
        visited, queue = set(), list(root_names)
        while queue:
            name = queue.pop()
            if name in visited:
                continue
            if name not in repo.taks:
                print(f"  WARNING: '{name}' not found in repo — skipping")
                continue
            visited.add(name)
            queue.extend(repo.graph.get(name, set()) - visited)
        return visited

    taks_to_rerun = _get_transitive_deps(repo, tak_ids)
    execution_subset = [t for t in repo.execution_order if t in taks_to_rerun]

    print(f"Partial re-run: {len(execution_subset)} TAKs (out of {len(repo.taks)} total)")
    for t in execution_subset:
        print(f"  - {t}")

    # Delete stale output rows (RawConcepts are never written, so skip them)
    taks_to_delete = [t for t in execution_subset if not isinstance(repo.get(t), RawConcept)]
    pattern_taks   = [t for t in taks_to_delete if isinstance(repo.get(t), Pattern)]

    conn = sqlite3.connect(str(DB_PATH))
    print()
    with tqdm(taks_to_delete, desc="Clearing OutputPatientData", unit="TAK") as pbar:
        for tak_name in pbar:
            pbar.set_postfix_str(tak_name)
            conn.execute("DELETE FROM OutputPatientData WHERE ConceptName = ?", (tak_name,))
    if pattern_taks:
        with tqdm(pattern_taks, desc="Clearing PatientQAScores  ", unit="TAK") as pbar:
            for tak_name in pbar:
                pbar.set_postfix_str(tak_name)
                conn.execute("DELETE FROM PatientQAScores WHERE PatternName = ?", (tak_name,))
    conn.commit()
    conn.close()

    patient_stats = mediator.process_all_patients_parallel(
        max_concurrent=16,
        patient_subset=patient_ids,
        tak_subset=execution_subset,
    )

else:
    # --- Full run ---
    patient_stats = mediator.run(
        max_concurrent=16,
        patient_subset=patient_ids,
    )

# ============================================================
# Aggregate stats
# ============================================================

assert 'patient_stats' in globals(), "patient_stats not found. Run the processing cell first."
assert 'repo' in globals(), "repo not found. Build the TAK repository first."

all_tak_names = sorted(repo.taks.keys())
agg_counts = defaultdict(int)
errors = []

for pid, stats in patient_stats.items():
    if not isinstance(stats, dict):
        continue
    # Top-level pipeline error
    if 'error' in stats and isinstance(stats['error'], str):
        errors.append((pid, '<pipeline>', stats['error']))
    for tak_name, val in stats.items():
        if tak_name == 'error':
            continue
        if isinstance(val, int):
            agg_counts[tak_name] += val
        elif isinstance(val, str) and val.startswith("ERROR"):
            errors.append((pid, tak_name, val))

# Build usage DataFrame
df_usage = pd.DataFrame(
    [(tak, int(agg_counts.get(tak, 0))) for tak in all_tak_names],
    columns=["TAK", "TotalRows"]
)

df_usage_sorted = df_usage.sort_values(by="TotalRows", ascending=False, ignore_index=True)

unused_taks = [tak for tak in all_tak_names if agg_counts.get(tak, 0) == 0]

# Summary (avoid surrogate escapes; use literal emoji or plain text)
print("\n" + "="*80)
print("📊 Aggregate TAK Usage Across Patients")
print("="*80)

total_rows_all = int(df_usage["TotalRows"].sum())
print(f"Total rows calculated: {total_rows_all} (including Raw Concepts)")
print(f"Total TAKs: {len(all_tak_names)}")
print(f"Used TAKs: {len(all_tak_names) - len(unused_taks)}")
print(f"Unused TAKs: {len(unused_taks)}")

# Top-N TAKs
top_n = 15
print(f"\n🏆 Top {top_n} TAKs by rows:")
for _, row in df_usage_sorted.head(top_n).iterrows():
    print(f"  • {row['TAK']}: {int(row['TotalRows'])} rows")

# Unused TAKs
if unused_taks:
    print("\n⚪ Unused TAKs (0 rows across all patients):")
    for i in range(0, len(unused_taks), 3):
        print("  " + ", ".join(unused_taks[i:i+3]))

# Errors grouped by TAK
if errors:
    print("\n❌ Errors by TAK (with patient IDs):")
    errs_by_tak = defaultdict(list)
    for pid, tak, msg in errors:
        errs_by_tak[tak].append((pid, msg))
    for tak, items in errs_by_tak.items():
        print(f"  • {tak}: {len(items)} errors")
        for pid, msg in items[:5]:
            print(f"      - patient {pid}: {msg}")

# Display full usage table
display(df_usage_sorted)

In [ ]:
# Query OutputPatientData
query = """
SELECT PatientId, ConceptName, StartDateTime, EndDateTime, Value
FROM OutputPatientData
"""

df_results = pd.read_sql_query(query, da.conn)
df_results = df_results.sort_values(by=["StartDateTime"], ignore_index=True)
df_results.head(10)

In [ ]:
df_results.to_csv("backend/data/temporal_data.csv", index=False)
print("\n✅ OutputPatientData saved to: backend/data/output_patient_data.csv")

In [ ]:
# Query OutputPatientData
query = """
SELECT PatientId, PatternName, StartDateTime, EndDateTime, ComplianceType, ComplianceScore
FROM PatientQAScores
"""
df_results = pd.read_sql_query(query, da.conn)
df_results = df_results.sort_values(by=["PatternName"], ignore_index=True)
df_results.head(10)

In [ ]:
df_results.to_csv("backend/data/qa_data.csv", index=False)
print("\n✅ QA Data saved to: backend/data/qa_data.csv")

# 🔍 TAK Debugging Toolkit

Interactive debugging for a single patient's abstraction flow.

**Features:**
- Load minimal data (only concepts relevant to selected TAKs)
- Use isolated debug DB (no production contamination)
- Full TAK execution trace with cache inspection
- Compare input → intermediate → output at each layer

In [ ]:
# ============================================================
# Configuration: Select Patient & TAKs to Debug
# ============================================================

DEBUG_PATIENT_ID = 1000

# Select TAKs to trace (in dependency order)
DEBUG_TAKS = [
    # Raw concepts
    "DIABETES_DIAGNOSIS",
    "ADMISSION",
    "GLUCOSE_MEASURE",
    "BASAL_BITZUA",
    "DEATH",
    "RELEASE",
    
    # Events
    "ADMISSION_EVENT",
    "DEATH_EVENT",
    "RELEASE_EVENT",
    
    # Contexts
    "DIABETES_DIAGNOSIS_CONTEXT",
    
    # Patterns
    "GLUCOSE_MEASURE_ON_ADMISSION_PATTERN"
]

# Debug DB path (separate from production)
DEBUG_DB_PATH = Path("backend/data/debug_mediator.db")

In [ ]:
# ============================================================
# Step 1: Extract Minimal Input Data for Debug Patient
# ============================================================

# Query production DB for this patient's input data
query = f"""
SELECT DISTINCT PatientId, ConceptName, StartDateTime, EndDateTime, Value
FROM InputPatientData
WHERE PatientId = {DEBUG_PATIENT_ID}
ORDER BY StartDateTime, ConceptName
"""

df_patient_input = pd.read_sql_query(query, da.conn)

print(f"📥 Input data for patient {DEBUG_PATIENT_ID}:")
print(f"   Total rows: {len(df_patient_input)}")
print(f"\n📊 Unique concepts in input:")
for concept in sorted(df_patient_input['ConceptName'].unique()):
    count = (df_patient_input['ConceptName'] == concept).sum()
    print(f"   - {concept}: {count} rows")

# Show sample
display(df_patient_input.head(10))

In [ ]:
# ============================================================
# Step 2: Create Debug DB and Load Minimal Data
# ============================================================

# Close any existing debug DB connections
if 'debug_da' in globals() and hasattr(debug_da, 'conn'):
    try:
        debug_da.conn.close()
        del debug_da
    except:
        pass

# Delete debug DB file entirely (force fresh start)
if DEBUG_DB_PATH.exists():
    import time
    for _ in range(3):  # Retry up to 3 times
        try:
            DEBUG_DB_PATH.unlink()
            print(f"🗑️  Deleted old debug DB: {DEBUG_DB_PATH}")
            break
        except PermissionError:
            time.sleep(0.5)  # Wait for file handles to release
    else:
        print(f"⚠️  Could not delete debug DB (file in use). Restart kernel and try again.")
        raise

# Create fresh debug DB
debug_da = DataAccess(db_path=str(DEBUG_DB_PATH), auto_create=True)
print(f"✅ Created fresh debug DB at {DEBUG_DB_PATH}")

# Load ONLY concepts relevant to DEBUG_TAKS
# Build list of input concepts needed by selected TAKs
mediator_temp = Mediator(knowledge_base_path=KB_PATH, data_access=debug_da)
repo_temp = mediator_temp.build_repository()

needed_concepts = set()
for tak_name in DEBUG_TAKS:
    tak = repo_temp.get(tak_name)
    if tak:
        if hasattr(tak, 'derived_from'):
            if isinstance(tak.derived_from, list):
                # Event/Context/Pattern
                for df in tak.derived_from:
                    needed_concepts.add(df['name'])
            else:
                # State/Trend
                needed_concepts.add(tak.derived_from)
        
        # Add attributes for raw-concepts
        if hasattr(tak, 'attributes'):
            for attr in tak.attributes:
                needed_concepts.add(attr['name'])

print(f"\n🎯 Concepts needed for selected TAKs:")
print(f"   {sorted(needed_concepts)}")

# Filter input data to only needed concepts
df_filtered = df_patient_input[df_patient_input['ConceptName'].isin(needed_concepts)].copy()

print(f"\n📦 Filtered input data:")
print(f"   Rows: {len(df_patient_input)} → {len(df_filtered)}")
print(f"   Concepts: {df_patient_input['ConceptName'].nunique()} → {df_filtered['ConceptName'].nunique()}")

# Load to debug DB
if not df_filtered.empty:
    df_filtered.to_sql('InputPatientData', debug_da.conn, if_exists='append', index=False)
    print(f"\n✅ Loaded {len(df_filtered)} rows to debug DB")
else:
    print(f"\n⚠️  No data to load (needed concepts not found in input)")

In [ ]:
# ============================================================
# Step 3: Run Mediator with TAK Cache Tracing
# ============================================================

# Initialize debug mediator
debug_mediator = Mediator(knowledge_base_path=KB_PATH, data_access=debug_da)
debug_repo = debug_mediator.build_repository()

# Enable detailed logging
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')

# Run processing for debug patient
print(f"\n🚀 Processing patient {DEBUG_PATIENT_ID} with cache tracing...\n")
print("="*80)

# Process patient synchronously (for easier debugging)
from multiprocessing import Manager
import threading
from core.mediator import _writer_worker

manager = Manager()
debug_write_queue = manager.Queue(maxsize=200)
writer_thread = threading.Thread(target=_writer_worker, args=(str(debug_mediator.db_path), debug_write_queue))
writer_thread.start()

stats, tak_cache = debug_mediator._process_patient_sync(DEBUG_PATIENT_ID, return_cache=True, write_queue=debug_write_queue)

debug_write_queue.put(None)
writer_thread.join()

print("\n" + "="*80)
print(f"\n✅ Processing complete!")
print(f"\n📊 Output stats:")
for tak_name, count in stats.items():
    if isinstance(count, int) and count > 0:
        print(f"   - {tak_name}: {count} rows")
    elif isinstance(count, str) and count.startswith("ERROR"):
        print(f"   ❌ {tak_name}: {count}")

print(f"\n🗄️  TAK Cache Size: {len(tak_cache)} TAKs cached")

In [ ]:
# ============================================================
# Step 4: Inspect TAK Cache (Intermediate Outputs)
# ============================================================

# Query cache for each selected TAK
print(f"\n🔍 TAK-by-TAK Cache Inspection:\n")

for tak_name in DEBUG_TAKS:
    tak = debug_repo.get(tak_name)
    if not tak:
        print(f"❌ {tak_name}: TAK not found in repository")
        continue
    
    # Get from cache (not DB!)
    if tak_name not in tak_cache:
        print(f"{'='*80}")
        print(f"📌 {tak_name} ({tak.family})")
        print(f"   ⚠️  NOT IN CACHE (TAK not processed or error occurred)")
        print()
        continue
    
    df_tak_output = tak_cache[tak_name]
    
    print(f"{'='*80}")
    print(f"📌 {tak_name} ({tak.family})")
    
    # Show derived_from info
    if hasattr(tak, 'derived_from'):
        if isinstance(tak.derived_from, list):
            deps = [d['name'] for d in tak.derived_from]
            print(f"   Derived from: {deps}")
        else:
            print(f"   Derived from: {tak.derived_from}")
    
    print(f"   Cached rows: {len(df_tak_output)}")
    
    if not df_tak_output.empty:
        print(f"\n   📊 Column dtypes:")
        for col, dtype in df_tak_output.dtypes.items():
            print(f"      {col}: {dtype}")
        
        print(f"\n   📋 Sample output (first 5 rows):")
        display(df_tak_output.head(5))
        
    else:
        print(f"\n   ⚠️  EMPTY OUTPUT")
        
        # Check dependencies to diagnose why
        if hasattr(tak, 'derived_from'):
            print(f"\n   🔍 Dependency Check:")
            if isinstance(tak.derived_from, list):
                for dep in tak.derived_from:
                    dep_name = dep['name']
                    if dep_name in tak_cache:
                        dep_rows = len(tak_cache[dep_name])
                        print(f"      - {dep_name}: {dep_rows} rows")
                    else:
                        print(f"      - {dep_name}: NOT IN CACHE")
            else:
                dep_name = tak.derived_from
                if dep_name in tak_cache:
                    dep_rows = len(tak_cache[dep_name])
                    print(f"      - {dep_name}: {dep_rows} rows")
                else:
                    print(f"      - {dep_name}: NOT IN CACHE")
    
    print()

In [ ]:
# ============================================================
# Step 5: Cleanup Debug DB
# ============================================================

# Close connection
debug_da.conn.close()

# Optionally delete debug DB
# DEBUG_DB_PATH.unlink()
# print(f"🗑️  Deleted debug DB: {DEBUG_DB_PATH}")

print(f"\n✅ Debug session complete!")
print(f"   Debug DB preserved at: {DEBUG_DB_PATH}")
print(f"   (Delete manually or re-run this block to recreate)")